# Module 5 — Phase stability, and the answer a flash cannot check

**2105603 Advanced Chemical Engineering Thermodynamics**
Department of Chemical Engineering, Chulalongkorn University
Soorathep Kheawhom

---

A flash calculation solves the equilibrium equations it was given. Those
equations have solutions that are not the global minimum of the Gibbs energy,
so a flash can converge cleanly, report no error, and be wrong.

This notebook does four things:

| step | question |
|---|---|
| 1 | build a feed where the flash is confidently wrong, and prove it |
| 2 | implement the tangent plane distance and use it as the test |
| 3 | compute a ternary liquid-liquid diagram and check it two ways |
| 4 | ask what a model can and cannot represent, before fitting it |

**Rule for this notebook, as for Module 4.** Every number you report must be
reproduced by an independent route. In this module that is unusually easy:
almost every stability result can be checked against a construction that shares
no code with it.

## 0. Setup

In [ ]:
import sys, subprocess, importlib, os

def ensure(pkg, pipname=None):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        pipname or pkg], check=False)

for p in ("numpy", "scipy", "matplotlib"):
    ensure(p)

if not os.path.isdir("vlekit"):
    import urllib.request, zipfile, io
    URL = "https://www.skhgroup.net/teaching/2105603/files/vlekit.zip"
    try:
        with urllib.request.urlopen(URL, timeout=30) as r:
            zipfile.ZipFile(io.BytesIO(r.read())).extractall(".")
        print("downloaded vlekit from the course page")
    except Exception as e:
        print("could not download vlekit:", e)

import numpy as np
import matplotlib.pyplot as plt
import vlekit as v
from vlekit import multi as M, flash as F, plots as pl

pl.use_style()
print("vlekit", v.__version__)

## 1. A flash that is confidently wrong

The cleanest counterexample is a binary, because the answer can be checked by a
construction you can draw. Take a symmetric mixture strong enough to split
($G^E/RT = A x_1 x_2$ with $A > 2$) and flash a feed in the middle of the
miscibility gap.

In [ ]:
A = 2.6
model = v.TwoSuffixMargules([A])
c1, c2 = v.component("benzene"), v.component("cyclohexane")   # carriers for Psat
z1, T = 0.50, 340.0

gap = F.miscibility_gap(model)
print(f"common-tangent binodal: x1 = {gap['x1_phase_a']:.4f} and "
      f"{gap['x1_phase_b']:.4f}")
print(f"spinodal:               x1 = {gap['spinodal'][0]:.4f} and "
      f"{gap['spinodal'][1]:.4f}")
print(f"the feed z1 = {z1} sits inside both.")

# the analytical check: for the symmetric case the binodal satisfies
#     ln[x/(1-x)] + A(1 - 2x) = 0
xa = gap["x1_phase_a"]
print(f"\nanalytical residual at the binodal: "
      f"{np.log(xa/(1-xa)) + A*(1-2*xa):.2e}")

Now ask for the bubble point of that feed. Nothing in a bubble-point
calculation knows what a miscibility gap is: it takes a liquid composition,
evaluates $\gamma$, and returns a pressure. It will do that for a liquid that
cannot exist, and it will not warn you.

In [ ]:
Pnaive, ynaive = v.bubble_P(model, np.array([z1]), np.array([T]), c1, c2)
print(f"bubble point asked for at x1 = {z1}: "
      f"P = {float(Pnaive[0]):.3f} kPa, y1 = {float(ynaive[0]):.4f}")
print("No warning. No error. The number is simply wrong.")

# What actually happens: the feed splits into the two conjugate liquids, and
# the system is a three-phase VLLE. The three-phase pressure can be had without
# solving a three-phase problem at all — each conjugate liquid must have the
# same bubble pressure, because they are in equilibrium with each other.
xa, xb = gap["x1_phase_a"], gap["x1_phase_b"]
Pa, ya = v.bubble_P(model, np.array([xa]), np.array([T]), c1, c2)
Pb, yb = v.bubble_P(model, np.array([xb]), np.array([T]), c1, c2)
print(f"\nliquid I  x1 = {xa:.4f} -> P = {float(Pa[0]):.3f} kPa, "
      f"y1 = {float(ya[0]):.4f}")
print(f"liquid II x1 = {xb:.4f} -> P = {float(Pb[0]):.3f} kPa, "
      f"y1 = {float(yb[0]):.4f}")
print(f"the two agree to {abs(float(Pa[0])-float(Pb[0])):.1e} kPa and "
      f"{abs(float(ya[0])-float(yb[0])):.1e} in y  <- that is the VLLE point")

err = 100 * (float(Pnaive[0]) - float(Pa[0])) / float(Pa[0])
print(f"\nnaive single-liquid bubble point: {float(Pnaive[0]):.3f} kPa")
print(f"the real three-phase pressure   : {float(Pa[0]):.3f} kPa")
print(f"error {err:+.2f} % — and a column model would carry it into every stage.")

Two things worth noticing.

The two conjugate liquids give the **same** bubble pressure and the **same**
vapour composition, to nine and eleven figures. Nothing in that calculation
solved a three-phase problem; the agreement is forced by the fact that the two
liquids are already in equilibrium with each other. That is the cheapest
possible route to a VLLE point, and it is a genuine check on the binodal.

The vapour composition also happens to be the same as the naive one. That is an
accident of the symmetric $G^E$ used here — $\ln(\gamma_1/\gamma_2)$ vanishes at
$x_1 = \tfrac12$ and at both binodal ends by symmetry — and it will not hold
for a real mixture. The **pressure** is the diagnostic; do not read anything
into $y$ agreeing.

## 2. The tangent plane distance

$$\mathrm{TPD}(\mathbf{w}) = \sum_i w_i\left[\ln w_i + \ln\gamma_i(\mathbf{w})
  - \ln z_i - \ln\gamma_i(\mathbf{z})\right]$$

Michelsen's criterion: the feed is stable if and only if this is non-negative
for **every** trial composition $\mathbf{w}$.

Write it yourself first. It is four lines, and typing them is the difference
between using a criterion and understanding it.

In [ ]:
class Binary(M.MultiModel):
    """Adapter so a binary model from vlekit.models works with vlekit.multi."""
    name = "binary"
    def __init__(self, m):
        super().__init__(2); self.m = m
    def ln_gamma(self, x):
        x = np.asarray(x, float)
        a, b = self.m.ln_gamma(np.array([x[0] / (x[0] + x[1])]))
        return np.array([a[0], b[0]])

mm = Binary(model)

def my_tpd(mm, z, w):
    """Write this yourself before running the cell below."""
    z = np.clip(np.asarray(z, float), 1e-300, None)
    w = np.clip(np.asarray(w, float), 1e-300, None)
    muz = np.log(z) + mm.ln_gamma(z)
    muw = np.log(w) + mm.ln_gamma(w)
    return float(np.dot(w, muw - muz))

z = np.array([z1, 1 - z1])
ws = np.linspace(0.01, 0.99, 99)
mine = np.array([my_tpd(mm, z, [w, 1 - w]) for w in ws])
lib  = np.array([M.tpd(mm, z, [w, 1 - w]) for w in ws])
print("max |mine - vlekit| =", np.max(np.abs(mine - lib)))
print("minimum TPD on this line:", mine.min(), "at w1 =", ws[mine.argmin()])

In [ ]:
st = M.stability(mm, z)
print(f"stable: {st['stable']},  min TPD {st['tpd_min']:+.5f}")
print(f"the phase that forms: w = {np.round(st['w'], 4)}")
print(f"\nthe common tangent said the phases are "
      f"{gap['x1_phase_a']:.4f} and {gap['x1_phase_b']:.4f}")
print("Two calculations that share no code, agreeing. That is the point.")

ts, curve = M.tpd_curve(mm, z)
fig, ax = pl.newfig(6.6, 4.2)
ax.axhline(0, color=pl.GRAPHITE, lw=1.0)
ax.plot(ts, curve, "-", color=pl.NAVY, lw=2.2)
ax.fill_between(ts, curve, 0, where=curve < 0, color=pl.RUST, alpha=0.20)
for xb in (gap["x1_phase_a"], gap["x1_phase_b"]):
    ax.axvline(xb, color=pl.TEAL, ls=":", lw=1.4)
ax.set_xlabel("trial composition $w_1$"); ax.set_ylabel("TPD")
ax.set_title("Negative anywhere means the feed is not a single stable phase")
plt.show()

**Question 2.1.** Move the feed to $z_1 = 0.05$ and re-run. The curve should
touch zero once and never go below. Where does it touch, and why is that point
not evidence of anything?

**Question 2.2.** Start the minimisation at $\mathbf{w} = \mathbf{z}$ and watch
it return zero. Explain in one sentence why a stability routine that did this
would report every feed as stable.

## 3. A ternary liquid-liquid diagram

Two components nearly immiscible, a third that distributes between them. This
is solvent extraction, and the tie-line slope is the whole economics of it.

The parameters below are **illustrative** — chosen to give a clean type-I dome,
not fitted to any measured ternary. If you have NRTL parameters from your
Module 4 dataset, substitute them and see what you get.

In [ ]:
tau = np.array([[0.00, 0.60, 3.20],
                [0.35, 0.00, 0.90],
                [2.80, 0.55, 0.00]])
alpha = np.array([[0.0, 0.30, 0.20],
                  [0.30, 0.0, 0.30],
                  [0.20, 0.30, 0.0]])
tern = M.NRTLMulti(tau, alpha)

print("Gibbs-Duhem residual :", tern.check_gibbs_duhem([0.3, 0.3, 0.4]))
print("pure-component limit :", tern.check_pure_limit())

feeds = [[(1 - t) * 0.5, t, (1 - t) * 0.5] for t in np.linspace(0.02, 0.58, 21)]
lines, dropped = M.tie_lines(tern, feeds)
print(f"\n{len(lines)} tie lines, {dropped} feeds did not split")

# two independent checks on every line
mb = max(np.max(np.abs(l["beta"] * l["xI"] + (1 - l["beta"]) * l["xII"]
                       - l["z"] / l["z"].sum())) for l in lines)
ia = max(np.max(np.abs(l["xI"] * np.exp(tern.ln_gamma(l["xI"]))
                       - l["xII"] * np.exp(tern.ln_gamma(l["xII"]))))
         for l in lines)
print(f"material balance closes to {mb:.2e}")
print(f"equal activities to        {ia:.2e}")
print("Neither is what the solver iterates on, so both are real checks.")

In [ ]:
pp, shortest = M.plait_point(lines)
print("plait point (extrapolated):", np.round(pp, 4),
      f"   shortest tie line computed: {shortest:.4f}")

fig, ax = pl.newfig(6.6, 5.8)
V = M.to_xy(np.eye(3))
ax.plot(np.append(V[:, 0], V[0, 0]), np.append(V[:, 1], V[0, 1]),
        "-", color=pl.GRAPHITE, lw=1.4)
for l in lines:
    p = M.to_xy(np.vstack([l["xI"], l["xII"]]))
    ax.plot(p[:, 0], p[:, 1], "-", color=pl.AMBER, lw=1.0, alpha=0.85)
a, b = M.binodal_from_tielines(lines)
for arr in (a, b):
    q = M.to_xy(arr)
    ax.plot(q[:, 0], q[:, 1], "o", color=pl.NAVY, ms=4, mfc="white", mew=1.2)
q = M.to_xy([pp])
ax.plot(q[0, 0], q[0, 1], "*", color=pl.TEAL, ms=18)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Type-I dome: every tie line is a converged flash")
plt.show()

**Question 3.1.** The tie lines are not parallel. Which phase does component 2
prefer, and how would you use that to choose a solvent?

**Question 3.2.** Increase $\tau_{13}$ and $\tau_{31}$ by 20 % and recompute.
Does the dome grow or shrink, and does the plait point move the way you
expected?

## 4. What the model can represent, before you fit it

Module 4 showed Wilson buying a fit with an infinite-dilution activity
coefficient of about twenty thousand. Here is the cause: Wilson's functional
form **cannot** produce a miscibility gap for any parameters at all.

This is a claim about a model, and a claim about a model can be tested.

In [ ]:
class WilsonTernary(M.MultiModel):
    name = "Wilson"
    def __init__(self, L):
        super().__init__(3); self.L = np.asarray(L, float)
    def ln_gamma(self, x):
        x = np.clip(np.asarray(x, float), 1e-300, None)
        s = self.L @ x
        return 1.0 - np.log(s) - np.array(
            [np.sum(x * self.L[:, i] / s) for i in range(3)])

rng = np.random.default_rng(0)
tested = split = 0
for _ in range(60):
    L = np.exp(rng.normal(0, 1.5, size=(3, 3))); np.fill_diagonal(L, 1.0)
    w = WilsonTernary(L)
    for zt in ([0.45, 0.10, 0.45], [1/3, 1/3, 1/3], [0.10, 0.45, 0.45]):
        tested += 1
        if not M.stability(w, zt)["stable"]:
            split += 1
print(f"Wilson: {split} splits out of {tested} random parameter sets and feeds")
print(f"NRTL  : the same feed splits -> "
      f"{not M.stability(tern, [0.45, 0.10, 0.45])['stable']}")

**Question 4.1.** This is an experiment, not a proof. Write the argument from
the functional form that explains the result — why can
$G^E/RT = -\sum_i x_i \ln(\sum_j \Lambda_{ij} x_j)$ never lose convexity?

**Question 4.2.** You are handed a dataset with a reported liquid-liquid split
and a paper that correlates it with Wilson parameters. What do you conclude
about the paper?

---

## Exercises

**E1. Your own system.** Take the NRTL parameters you fitted in the Module 4
notebook and run the stability test across the whole composition range. Report
whether your fit predicts a split, at what compositions, and — the important
part — whether any data you fitted could have told you.

**E2. The metastable band.** For the binary in Section 1, compute the binodal
and the spinodal, then compute the height of the barrier a fluctuation must
climb from a composition halfway between them. Compare it with $RT$ and say
what you would expect to observe in a beaker.

**E3. From VLE to LLE.** Fit NRTL to a VLE dataset, then use those parameters
to predict mutual solubilities. Find a literature solubility for the same pair
and compare. Report the error as a factor, not a percentage.

**E4. Tie-line slope.** For the ternary in Section 3, compute the distribution
coefficient $K_2 = x_2^{\rm I}/x_2^{\rm II}$ along the dome. Where is
extraction most efficient, and what happens near the plait point?

**E5. Residue curves.** Integrate residue curves for the acetone / chloroform /
methanol set used in the lecture, locate the boundary by bisection, and place
two feeds either side. State the products each one reaches.

---

## AI checkpoint

Ask a language model: *"Will a mixture of 45 % component 1, 10 % component 2 and
45 % component 3 split into two liquid phases?"* — giving it the NRTL parameters
from Section 3.

Then run the stability test.

This is a question that **sounds like recall and is actually a computation**.
Record what the model said, what the computation said, and — if they agreed —
whether the model gave any reason to believe it had done the calculation rather
than pattern-matched a plausible answer. Change the feed to one just outside the
dome (try $x_3 = 0.004$) and ask again. A model that gets both right by
reasoning will change its answer; one that is pattern-matching usually will not.

---

*vlekit and this notebook: Soorathep Kheawhom, 2105603, Chulalongkorn
University.*